# prometheus_client: counters, gauges, histograms, summaries and enums

In [ ]:
# Example 1: Counter
# Each example uses its own CollectorRegistry, so metric names never clash with
# metrics registered elsewhere in the same process.
from prometheus_client import CollectorRegistry, Counter, generate_latest

registry = CollectorRegistry()
c = Counter('my_counter', 'An example counter', registry=registry)
c.inc()
assert registry.get_sample_value('my_counter_total') == 1.0
c.inc(2.5)
value = registry.get_sample_value('my_counter_total')
print(f"my_counter_total = {value}")
assert value == 3.5, value

# Counters only go up.
try:
    c.inc(-1)
except ValueError as e:
    print(f"Negative increment rejected: {e}")
else:
    raise AssertionError("Counter accepted a negative increment")

# Labelled counter: one time series per label value
requests = Counter('http_requests', 'HTTP requests', ['method'], registry=registry)
requests.labels(method='GET').inc()
requests.labels(method='GET').inc()
requests.labels(method='POST').inc()
assert registry.get_sample_value('http_requests_total', {'method': 'GET'}) == 2.0
assert registry.get_sample_value('http_requests_total', {'method': 'POST'}) == 1.0

exposition = generate_latest(registry).decode()
print(exposition)
assert 'my_counter_total 3.5' in exposition, exposition
assert 'http_requests_total{method="GET"} 2.0' in exposition, exposition


In [ ]:
# Example 2: Gauge
from prometheus_client import CollectorRegistry, Gauge, generate_latest

registry = CollectorRegistry()
g = Gauge('my_gauge', 'An example gauge', registry=registry)
g.set(5)
assert registry.get_sample_value('my_gauge') == 5.0
g.inc(2)
g.dec(3)
value = registry.get_sample_value('my_gauge')
print(f"my_gauge = {value}")
assert value == 4.0, value
assert 'my_gauge 4.0' in generate_latest(registry).decode()


In [ ]:
# Example 3: Histogram
from prometheus_client import CollectorRegistry, Histogram, generate_latest

registry = CollectorRegistry()
h = Histogram('my_histogram', 'An example histogram', buckets=(0.1, 1.0, 5.0), registry=registry)
h.observe(0.5)
h.observe(2.0)

total = registry.get_sample_value('my_histogram_sum')
count = registry.get_sample_value('my_histogram_count')
print(f"my_histogram_sum = {total}, my_histogram_count = {count}")
assert total == 2.5, total
assert count == 2.0, count
# Buckets are cumulative: le="1.0" counts 0.5, le="5.0" and "+Inf" count both.
assert registry.get_sample_value('my_histogram_bucket', {'le': '0.1'}) == 0.0
assert registry.get_sample_value('my_histogram_bucket', {'le': '1.0'}) == 1.0
assert registry.get_sample_value('my_histogram_bucket', {'le': '5.0'}) == 2.0
assert registry.get_sample_value('my_histogram_bucket', {'le': '+Inf'}) == 2.0
assert 'my_histogram_sum 2.5' in generate_latest(registry).decode()


In [ ]:
# Example 4: Summary
import math
from prometheus_client import CollectorRegistry, Summary, generate_latest

registry = CollectorRegistry()
s = Summary('my_summary', 'An example summary', registry=registry)
s.observe(0.7)
s.observe(1.3)

total = registry.get_sample_value('my_summary_sum')
count = registry.get_sample_value('my_summary_count')
print(f"my_summary_sum = {total}, my_summary_count = {count}")
assert math.isclose(total, 2.0), total
assert count == 2.0, count

# time() records the duration of the block it wraps.
with s.time():
    pass
assert registry.get_sample_value('my_summary_count') == 3.0
assert 'my_summary_count 3.0' in generate_latest(registry).decode()


In [ ]:
# Example 5: Enum
from prometheus_client import CollectorRegistry, Enum, generate_latest

print("Prometheus Enum Metric Demo")

registry = CollectorRegistry()
app_states = ['starting', 'running', 'maintenance', 'stopping']
health_states = ['healthy', 'degraded', 'unhealthy']
app_state = Enum('application_state', 'Current application state',
                 states=app_states, registry=registry)
service_health = Enum('service_health', 'Service health status',
                      states=health_states, registry=registry)

def current(name, states):
    """The state whose sample is 1; every other state's sample is 0"""
    values = {st: registry.get_sample_value(name, {name: st}) for st in states}
    assert sorted(values.values()) == [0.0] * (len(states) - 1) + [1.0], values
    return next(st for st, v in values.items() if v == 1.0)

# An Enum starts in its first state.
assert current('application_state', app_states) == 'starting'

steps = [('running', 'healthy'), ('maintenance', 'degraded'), ('running', 'healthy'), ('stopping', 'unhealthy')]
for i, (state, health) in enumerate(steps, 1):
    app_state.state(state)
    service_health.state(health)
    print(f"Step {i}: App={state}, Health={health}")
    assert current('application_state', app_states) == state
    assert current('service_health', health_states) == health

# Setting a state that was not declared is an error.
try:
    app_state.state('crashed')
except ValueError as e:
    print(f"Unknown state rejected: {e}")
else:
    raise AssertionError("Enum accepted an undeclared state")

exposition = generate_latest(registry).decode()
assert 'application_state{application_state="stopping"} 1.0' in exposition, exposition
print(f"\nFinal app state: {current('application_state', app_states)}")
print(f"Final health state: {current('service_health', health_states)}")
